In [1]:
import zmq
import threading
import logging

In [2]:
class RealtimeSubWorker:
    def __init__(self, aEndPtSrc, aEndPtRep = None, aTopic=""):
        self._topic_skip = len(aTopic)
        
        self._context = zmq.Context()
        self._socket = self._context.socket(zmq.SUB)
        self._socket.connect(aEndPtSrc)
        if aEndPtRep is not None:
            self._socket.connect(aEndPtRep)
        self._socket.setsockopt_string(zmq.SUBSCRIBE,  aTopic)

        self._stop_event = threading.Event()
        self._thread = None
        

    def _on_message(self, aMsg):
        raise NotImplementedError("Subclasses should implement this method!")

    
    def start_worker(self):
        def run():
            try:
                while not self._stop_event.is_set():
                    try:
                        aMsg = self._socket.recv_string(flags=zmq.NOBLOCK)
                        self._on_message(aMsg[self._topic_skip:])  # strip out the message topic prefix
                    except zmq.Again:
                        pass  # No message received, continue the loop
            finally:
                self._socket.close()
                self._context.term()
        
        self._thread = threading.Thread(target=run)
        self._thread.start()

    
    def _close(self):  #give subclass a chance to clean up resources
        raise NotImplementedError("Subclasses should implement this method!")

    
    def stop_worker(self):
        self._stop_event.set()
        self._thread.join()

        self._close()
        del self